# 🏗️ 쿠팡 물류창고 관제 시스템 핵심 코드 및 알고리즘 종합 가이드

본 Jupyter Notebook은 다중 로봇 물류창고 관제 시스템(Control Tower)의 4대 핵심 컴포넌트(**Docker 인프라, PostgreSQL DB 스키마, ROS 2 Control Tower 노드, FastAPI 대시보드**)의 상세 설계 사양, 핵심 알고리즘 및 시퀀스 흐름도(Mermaid Flowchart)를 종합 해설하는 **엔지니어링 가이드북**입니다.

---

## 🐳 1. Docker 기반 하이브리드 DB 인프라 (`docker-compose.yml`)

관제 시스템의 데이터 저장소는 성능 극대화 및 데이터 무결성 보장을 위해 **관계형 DB(PostgreSQL)**와 **인메모리 DB(Redis)**를 혼합하여 운용합니다.

### 📌 1.1 하이브리드 설계 아키텍처 및 핵심 이유

```mermaid
graph TD
    AMR[AMR 주행 제어기] -->|고주파 텔레메트리 10Hz| Redis[(Redis 캐시)]
    Sorter[적재/포장 로봇] -->|트랜잭션 데이터 1Hz| SQL[(PostgreSQL)]
    CT[Control Tower] <-->|ZSET 우선순위 스케줄링| Redis
    CT <-->|조인 및 이력 보존| SQL
    Dash[FastAPI Dashboard] <-- WebSocket (1.5s) --> Browser[웹 브라우저]
```

* **PostgreSQL (Port 5432)**:
  * **역할**: 마스터 데이터(로봇 목록, 바닥 격자 맵 정보) 관리 및 영구 트랜잭션 데이터(패키지 라이프사이클, 작업대 보관 이력) 보존.
  * **키 포인트**: 외래키(Foreign Key) 제약 조건을 활용해 작업대가 이동하더라도 상자의 적재 상태 정보가 불일치하지 않도록 보장.
* **Redis (Port 6379)**:
  * **역할**: 고주파수(10Hz+) AMR 상태 데이터 캐싱 및 Sorted Set(ZSET) 기반 우선순위 제어 명령 큐 관리.
  * **키 포인트**: 초당 수십 번 발생하는 AMR의 위치 좌표 갱신 연산이 디스크 I/O 병목을 유발하지 않도록 인메모리 처리하여 관제탑 성능을 30 FPS 이상 유지.

### 📌 1.2 데이터베이스 커넥션 풀 (Connection Pool) 알고리즘
* **문제점**: ROS 2는 멀티스레드 비동기 콜백 환경입니다. 각 스레드가 쿼리를 실행할 때마다 DB 연결을 맺고 끊으면 오버헤드가 발생하고 포트 고갈 현상이 생깁니다.
* **해결 알고리즘**: `psycopg2.pool.ThreadedConnectionPool`을 도입하여 일정 수(예: 최소 5개, 최대 20개)의 DB 커넥션을 미리 생성해 둡니다. 스레드가 쿼리를 요청하면 풀에서 유휴 커넥션을 빌려준 뒤, 처리가 끝나면 원자적으로 풀에 반환합니다.
### 📌 1.3 SQL 컨테이너 관리 및 영속성 (Persistence)
* **데이터 영속성**: 호스트 볼륨 매핑 및 외부 볼륨 폴더 매핑을 통해 컨베이어나 도커 컨테이너가 재시작되더라도 실시간 적재/이송 로그가 유실되지 않도록 보장.
* **Adminer GUI (Port 8082)**: PostgreSQL 데이터베이스의 스키마 구조, 테이블 관계, 인덱스 성능 등을 웹 브라우저에서 실시간으로 시각화 및 관리 가능.


In [ ]:
# Docker 컨테이너의 기동 상태 및 포트 매핑을 확인합니다.
!docker ps --format "table {{.Names}}\t{{.Ports}}\t{{.Status}}"

---
### 📌 2.4 SQL 테이블 관계 및 성능 인덱싱 (Indexing)
* **테이블 관계**: `packages` (택배)가 `workstations` (작업대)를 참조하고, `workstations`가 `warehouse_locations` (창고 스팟)과 `floor_qr_map` (물리 좌표 마커)를 참조하는 외래키 관계망 형성.
* **성능 인덱스 도입**: 대용량 CSV 입고(예: 150개 이상) 및 실시간 조회 시 Full Table Scan 병목을 방지하기 위해 5대 핵심 인덱스(`idx_packages_status`, `idx_packages_route_zone`, `idx_packages_workstation`, `idx_workstations_location`, `idx_floor_qr_location`) 적용.


## 🗄️ 2. PostgreSQL 스키마 설계 및 이월 적재 초기화 (`init.sql` & `init_june_8th_state.py`)

데이터 정규화 및 영업일 전환(Day Transition)을 실현하는 DB 스키마 구조와 데이터 흐름 알고리즘입니다.

### 📌 2.1 데이터 정규화 (1:N 조인 설계) 키 포인트
* 기존에는 작업대 테이블(`workstations`)에 8개 슬롯 정보를 직접 컬럼(예: `slot_1_status` 등)으로 넣어 공간 낭비와 불일치 위험이 있었습니다.
* 개선 후에는 `packages` 테이블이 **`workstation_id` (외래키)**와 **`slot_number` (1~8)**를 소유하여, 해당 작업대에 종속된 패키지를 `JOIN` 형태로 쿼리합니다. 이로써 슬롯 개수가 변경(예: 6개, 12개)되어도 테이블 스키마를 변경할 필요가 없습니다.

### 📌 2.2 관계형 데이터베이스 테이블 구조 상세 정의 (DB 스키마)
본 관제 시스템은 아래와 같이 총 5개의 테이블로 구성되어 실시간 물류와 로봇의 상태 정보를 관리합니다.

#### ① 로봇 정보 테이블 (`robots`)
관제 센터가 제어하는 모든 물류 로봇의 정보와 물리 QR코드 식별자의 매핑 테이블입니다.
* **Primary Key**: `robot_id`
* **컬럼 구조**:
  * `robot_id` (`VARCHAR(50)`): 로봇 고유 문자열 ID (예: `'bg2'`, `'sg2_in_01'`)
  * `robot_type` (`VARCHAR(50)`): 로봇의 분류/역할군 (예: `'CONVEYOR_SORTER'`, `'MANIPULATOR'`)
  * `qr_id` (`VARCHAR(100)`): 로봇 고유 QR코드 ID (예: `'ROBOT_bg2'`, `'ROBOT_sg2_in_01'`)

#### ② 작업대 정보 테이블 (`workstations`)
로봇들이 상자를 싣는 2x4 슬롯 기반 작업대의 실시간 물리적 위치와 식별 QR코드 정보를 관리합니다.
* **Primary Key**: `workstation_id`
* **컬럼 구조**:
  * `workstation_id` (`VARCHAR(50)`): 작업대 고유 문자열 ID (예: `'WS01'`, `'WS10'`)
  * `current_location` (`VARCHAR(50)`): 작업대의 실시간 위치 (예: `'sg2_in_01_A'`, `'spot_01'`, `'sg2_out_00_A'`)
  * `qr_id` (`VARCHAR(100)`): 작업대 고유 QR코드 ID (예: `'WORKSTATION_WS01'`, `'WORKSTATION_WS10'`)
  * `status` (`VARCHAR(50)`): 작업대의 제어 상태 (예: `'WAITING'`, `'PROCESSING'`)
  * `reserved_by` (`VARCHAR(50)`): 현재 작업대를 예약/선점 중인 AMR 식별자 (예: `'AMR_01'`, `NULL`)

#### ③ 창고 세부 스팟 관리 테이블 (`warehouse_locations`)
창고 내부의 개별 보관 슬롯 구역들의 점유 현황과 주차된 작업대 매핑을 관리합니다.
* **Primary Key**: `spot_id`
* **Foreign Key**: `workstation_id` (작업대 테이블 참조)
* **컬럼 구조**:
  * `spot_id` (`VARCHAR(50)`): 창고 내 고유 주차 구역 ID (예: `'spot_01'`, `'spot_10'`)
  * `workstation_id` (`VARCHAR(50)`): 주차된 작업대 고유 ID (비었을 시 `NULL`)
  * `status` (`VARCHAR(20)`): 스팟 점유 상태 (`EMPTY` / `OCCUPIED`)

#### ④ 택배 정보 테이블 (`packages`)
입고되는 모든 택배의 상태 및 적재/출고 이력을 관리하는 데이터의 흐름 핵심 테이블입니다.
* **Primary Key**: `package_id`
* **Foreign Key**: `workstation_id` (작업대 테이블 참조)
* **컬럼 구조**:
  * `package_id` (`VARCHAR(50)`): 상자 바코드 또는 고유 ID (예: `'PKG_20260608_001'`)
  * `customer_name` (`VARCHAR(100)`): 택배 수령인 성함 (예: `'김철수'`)
  * `route_zone` (`VARCHAR(20)`): 분류 배송 예정 날짜 (예: `'2026-06-08'`)
  * `status` (`VARCHAR(50)`): 진행 상태 (예: `'WAITING'`, `'IN_WORKSTATION'`, `'IN_WAREHOUSE'`, `'COMPLETED'`)
  * `outbound_id` (`VARCHAR(100)`): 포장 후 출고 고유 바코드
  * `workstation_id` (`VARCHAR(50)`): 적재된 작업대 ID
  * `slot_number` (`INT`): 작업대 내 적재 슬롯 번호 (1~8)
  * `qr_id` (`VARCHAR(100)`): 택배 고유 QR코드 ID

#### ⑤ 공간 바닥 QR코드 격자 맵 테이블 (`floor_qr_map`)
AMR의 3D 공간 자율주행 및 위치 좌표 해석(Localization)을 위해 바닥에 매핑된 QR코드 격자 맵 정보를 관리합니다.
* **Primary Key**: `qr_id`
* **컬럼 구조**:
  * `qr_id` (`VARCHAR(100)`): 바닥 QR코드 고유 ID (예: `'FLOOR_X_1.5_Y_3.0'`)
  * `x_coord` (`DOUBLE PRECISION`): 물리 X 좌표 (m)
  * `y_coord` (`DOUBLE PRECISION`): 물리 Y 좌표 (m)
  * `z_coord` (`DOUBLE PRECISION`): 물리 Z 좌표 (m)
  * `location_name` (`VARCHAR(50)`): 매핑되는 논리적 위치명 (예: `'spot_01'`, `'sg2_in_01_A'`)
  * `location_type` (`VARCHAR(50)`): 위치 용도 분류 (예: `'PARKING_SPOT'`, `'PATHWAY'`)
  * `description` (`TEXT`): 세부 위치 설명

### 📌 2.3 데이터베이스 테이블 ERD 관계도
```mermaid
erDiagram
    WORKSTATIONS ||--o{ PACKAGES : "contains"
    WORKSTATIONS ||--o| WAREHOUSE_LOCATIONS : "parked at"
    ROBOTS {
        VARCHAR robot_id PK
        VARCHAR robot_type
        VARCHAR qr_id UNIQUE
    }
    WORKSTATIONS {
        VARCHAR workstation_id PK
        VARCHAR current_location
        VARCHAR qr_id UNIQUE
        VARCHAR status
        VARCHAR reserved_by
    }
    WAREHOUSE_LOCATIONS {
        VARCHAR spot_id PK
        VARCHAR workstation_id FK
        VARCHAR status
    }
    PACKAGES {
        VARCHAR package_id PK
        VARCHAR customer_name
        VARCHAR route_zone
        VARCHAR status
        VARCHAR outbound_id
        VARCHAR workstation_id FK
        INT slot_number
        VARCHAR qr_id UNIQUE
    }
    FLOOR_QR_MAP {
        VARCHAR qr_id PK
        DOUBLE x_coord
        DOUBLE y_coord
        DOUBLE z_coord
        VARCHAR location_name
        VARCHAR location_type
        TEXT description
    }
```

### 📌 2.4 영업일 전환 및 이월 적재 (Day Transition & Carry-over) 알고리즘

```mermaid
sequenceDiagram
    participant CT as Control Tower (관제탑)
    participant DB as PostgreSQL
    participant RD as Redis

    Note over CT, RD: 오늘 영업일 완료 (오늘 날짜 미완료 패키지 = 0)
    CT->>DB: Daily Report 마크다운 보고서 자동 생성
    CT->>RD: system:day_status를 'PENDING_TRANSITION'으로 변경
    Note over CT: 관리자가 대시보드에서 다음 날 영업 시작 클릭
    CT->>RD: system:today_date를 하루 뒤 날짜로 업데이트
    CT->>DB: [이월 재배치 예약] 실행
    CT->>DB: 2번 라인 작업대 -> 1번 라인 Active 버퍼로 이송 예약
    CT->>DB: 3번 라인 작업대 -> 2번 라인 Active 버퍼로 이송 예약
    CT->>DB: 3번 라인에는 메인 창고 주차장 스팟에서 새 빈 작업대 공급
    Note over CT, DB: 이월 작업대는 기존 적재 슬롯(예: 5/8)부터 신규 상자를 누적 적재
```

### 📌 2.5 6월 8일 데모용 초기 상태 맵 데이터 검증
초기화 실행 시 `WS01`은 이미 8개 상자가 완충되어 출고 대기 구역(`stage_01`)에 존재하고, `WS02`(1번 라인 Active), `WS03`(2번 라인 Active)에는 각각 전날 쌓다 남은 **5개의 이월 상자**들이 적재된 채로 대기합니다.

In [ ]:
import psycopg2
import os

db_host = os.environ.get("POSTGRES_HOST", "localhost")
try:
    conn = psycopg2.connect(
        host=db_host,
        database="warehouse_db",
        user="rokey",
        password="rokey_pass",
        port=5432
    )
    cursor = conn.cursor()
    
    # 1. 각 라인별 작업대와 슬롯 적재 현황 조회 (조인 쿼리)
    query = """
    SELECT w.workstation_id, w.current_location, COUNT(p.package_id) as loaded_slots
    FROM workstations w
    LEFT JOIN packages p ON w.workstation_id = p.workstation_id AND p.status IN ('IN_WORKSTATION', 'IN_WAREHOUSE')
    GROUP BY w.workstation_id, w.current_location, w.status
    ORDER BY w.workstation_id;
    """
    cursor.execute(query)
    print("📊 작업대별 적재 상태 검증:")
    for row in cursor.fetchall():
        print(f"   * 작업대: {row[0]} | 위치: {row[1]:<15} | 적재된 슬롯 수: {row[2]}/8 개")
        
    cursor.close()
    conn.close()
except Exception as e:
    print(f"❌ DB 조회 실패: {e}")

---
### 📌 3.4 Control Tower 내 SQL 세션 및 트랜잭션 관리
* **스레드 세이프 커서 락 (`threading.RLock`)**: ROS 2 `MultiThreadedExecutor` 하에서 다중 비동기 콜백 스레드가 동시에 단일 PostgreSQL 커넥션 커서에 접근해 데이터가 충돌하지 않도록 재진입 가능 락(`RLock`)으로 SQL 질의 블록을 보호.
* **장애 발생 롤백 메커니즘 (`recover_workstation_move_db_state`)**: AMR과의 ROS 2 Action 연결 실패나 주행 오류 발생 시, 작업대의 위치가 이동 중(`MOVING_TO_...`) 상태로 교착되어 WMS가 꼬이지 않도록 작업대 및 주차 스팟 상태를 원래대로 자동 되돌리는 DB 롤백 트랜잭션 처리.


## 🤖 3. ROS 2 Control Tower 노드 (`control_tower_node.py`)

관제탑 노드는 물류 창고 내 다중 AMR(자율이송로봇)과 적재/포장 매니퓰레이터 로봇들의 실시간 제어, DB 상태 동기화 및 인터로킹(Interlocking)을 총괄하는 **중앙 관제 허브**입니다. 
본 절에서는 관제탑 노드가 공정을 스케줄링하고 명령을 발행하는 세부 아키텍처와 핵심 알고리즘 및 규칙을 다룹니다.

---

### 📌 3.1 컨트롤 타워 핵심 아키텍처 및 테스크 매니저 구조

관제탑 노드는 비동기 멀티스레드 콜백 실행기(`MultiThreadedExecutor`)를 사용하여 다음과 같은 핵심 스레드 및 관리 루프들을 병렬로 운용합니다.

#### ① 모니터링 매니저 (`check_completed_workstations`)
* **기동 주기**: 1.5초 타이머 콜백
* **역할**: WMS 데이터베이스(PostgreSQL)의 작업대 및 패키지 상태를 상시 스캔하여, 신규 이송이 필요한 작업대를 감지하고 태스크를 자동 생성해 명령 큐에 푸시합니다.
* **감지 규칙**:
  * **입고 완충 감지**: 입고라인 Active 구역(`sg2_in_XX_A`)에 위치한 작업대의 적재 패키지 수량이 **8개**에 도달하면 해당 작업대 인출 태스크(`RETRIEVE_FULL_WORKSTATION`)와 빈 작업대 공급 태스크(`SUPPLY_EMPTY_WORKSTATION`)를 동시에 생성합니다.
  * **출고 요청 감지**: 출고라인 포장대(`sg2_out_00_A`)가 비어 있고(Empty), 보관 창고 또는 대기 구역에 오늘 배송 마감 날짜(`route_zone`)에 해당하는 패키지를 담은 작업대가 존재할 경우 해당 작업대의 출고 이송 태스크(`PRE_FETCH_WORKSTATION`)를 생성합니다.
  * **출고 완료 감지**: 출고라인 포장대에서 모든 패키지의 포장 작업이 완료(`status = 'COMPLETED'`)되면 빈 작업대 반납 태스크(`RETURN_EMPTY_WORKSTATION`)를 생성합니다.

#### ② 스케줄러 매니저 (`task_scheduler_loop`)
* **기동 주기**: 1.0초 타이머 콜백
* **역할**: Redis Sorted Set 우선순위 큐(`queue:amr_tasks`)를 모니터링하여, 적절한 가용 AMR에게 최적 매핑 후 이송 명령(Action)을 하달합니다.
* **핵심 제어 규칙**:
  * **동시 가동 제한 (Fleet Mutex)**: 창고 주행 통로의 병목 및 주행 데드락(교착) 방지를 위해 동시 구동 중인 AMR 대수(`active_amr_tasks`)를 **최대 3대**로 강제 제한합니다. 카운터가 3 이상인 경우 추가 배정을 중단하고 큐에서 대기시킵니다.
  * **리소스 락 (Resource Lock)**: 동일한 작업대 ID 또는 동일한 목표 주차 위치(`target_location`)에 대해 이중 배정이 일어나 충돌이 발생하는 것을 원천 차단하기 위해, 태스크를 팝하기 전 `is_workstation_or_target_busy` 함수를 통해 대상 자원의 점유 여부를 검증합니다.

---

### 📌 3.2 Redis ZSET 기반 우선순위 큐 스케줄링 규칙

모든 이송 태스크는 긴급도와 작업 흐름 상의 선후 관계를 고려하여 가중치(Priority Score)를 부여받아 Redis ZSET에 삽입됩니다.

| 우선순위 등급 | 태스크 종류 (Task Type) | 가중치 점수 (Score) | 우선순위 결정 규칙 및 발생 조건 |
| :---: | :--- | :---: | :--- |
| **1순위 (최우선)** | `ROTATE_WORKSTATION`<br>`RETRIEVE_FULL_WORKSTATION`<br>`PRE_FETCH_WORKSTATION`<br>`DIRECT_WAREHOUSE` | **100** | **생산 라인 중단 방지**: 입/출고 매니퓰레이터의 작업 일시정지(Pause) 시간을 최소화하기 위한 180도 회전, 완충 작업대 신속 인출, 출고 포장 전용 작업대 우선 공급 및 긴급 직입고 태스크. |
| **2순위 (중간)** | `REARRANGE_TO_WAREHOUSE` | **50** | **이월 공정 정리**: 영업 마감 후 내일/모레 출고분 작업대를 보관 창고로 이송하여 입고 라인 Active 스팟을 비우는 정리 작업. |
| **3순위 (최하)** | `SUPPLY_EMPTY_WORKSTATION`<br>`RETURN_EMPTY_WORKSTATION` | **20** | **자원 회수 및 준비**: 빈 작업대를 대기 구역에서 회수하거나 신규 빈 작업대를 입고 라인에 사전 대기시키는 비긴급 보조 작업. |

---

### 📌 3.3 JIT (Just-In-Time) 일시정지 및 인터로킹(Interlocking) 알고리즘

매니퓰레이터 로봇(적재/포장)과 자율주행 AMR 간의 안전한 협업과 시뮬레이터 상의 물리 붕괴(오브젝트 낙하/충돌) 방지를 위해 하드웨어 인터로킹 프로토콜을 사용합니다.

#### 🔄 JIT 제어 인터로킹 시퀀스 규칙:
1. **4슬롯 적재/포장 완료 (180도 회전 시점)**:
   - 로봇이 4번째 상자 적재(입고) 또는 포장(출고) 완료 시 관제탑으로 `ReportInboundProgress` 호출 또는 피드백 전송.
   - 관제탑은 즉시 `/{robot_id}/pause_status` 토픽에 `True`를 발행하여 로봇을 일시정지시킵니다.
   - 관제탑은 큐에 `ROTATE_WORKSTATION` 태스크를 발행하여 AMR에게 180도 회전을 명령합니다.
   - AMR이 제자리 회전을 성공적으로 마쳐 `workstation_move_completed_callback`이 트리거되면, 관제탑은 로봇의 일시정지 토픽에 `False`를 발행하여 5~8번째 슬롯의 작업을 안전하게 재개하도록 잠금을 해제합니다.
2. **8슬롯 적재 완료 (작업대 만석 교체 시점)**:
   - 8번째 상자 적재 즉시 로봇을 일시정지(`True`)하고, 완충 작업대를 인출(`RETRIEVE`)한 뒤 새 빈 작업대를 안착(`SUPPLY`)시킵니다.
   - 새 빈 작업대가 무사히 도킹 완료되는 즉시 일시정지를 해제(`False`)하여 적재를 다시 이어나갑니다.

---

### 📌 3.4 최단 거리 기반 AMR 최적 매핑 알고리즘

* **수식**: Euclidean Distance $d = \sqrt{(x_{start} - x_{amr})^2 + (y_{start} - y_{amr})^2}$
* **배정 메커니즘**:
  1. 배정할 태스크의 출발지(예: `sg2_in_01_A`)의 3D 공간 물리 좌표 $(x_{start}, y_{start})$를 PostgreSQL DB의 `floor_qr_map` 테이블에서 획득합니다.
  2. Redis 캐시에 등록된 모든 AMR 중 현재 상태가 `'IDLE'` 이면서 가동 가능한(`available = true`) AMR들의 실시간 좌표 $(x_{amr}, y_{amr})$를 조회합니다.
  3. 출발지 좌표와의 유클리드 거리가 최소($d$)인 최단거리 AMR을 최적의 적합 로봇으로 선정하여 ROS 2 Action 이송 명령을 하달합니다.

---

### 📌 3.5 Control Tower 전체 의사결정 및 제어 흐름도

#### ① 모니터링 매니저 의사결정 및 스케줄러 루프 흐름도
```mermaid
graph TD
    Start([1.5초 주기 모니터링 시작]) --> QueryDB[WMS PostgreSQL DB 상태 조회]
    QueryDB --> CheckInbound{입고 Active 작업대<br>상자 적재 8칸 완충?}
    
    CheckInbound -->|Yes| PushFull[Redis ZSET에 인출/공급 태스크 생성<br>RETRIEVE_FULL & SUPPLY_EMPTY / Priority: 100 & 20]
    CheckInbound -->|No| CheckOutbound{출고 포장대 sg2_out_00_A<br>비어있고 대기 물량 존재?}
    
    CheckOutbound -->|Yes| PushPrefetch[Redis ZSET에 출고 준비 태스크 생성<br>PRE_FETCH_WORKSTATION / Priority: 100]
    CheckOutbound -->|No| CheckEmptyWS{출고 포장대 작업대<br>모든 포장 완료 COMPLETED?}
    
    CheckEmptyWS -->|Yes| PushReturn[Redis ZSET에 작업대 반납 태스크 생성<br>RETURN_EMPTY_WORKSTATION / Priority: 20]
    CheckEmptyWS -->|No| End[모니터링 종료 및 대기]
    
    PushFull --> End
    PushPrefetch --> End
    PushReturn --> End
    
    subgraph SchedulerLoop [1.0초 주기 스케줄러 실행 루프]
        CheckLimit{실행 중인 AMR 태스크<br>active_amr_tasks >= 3?}
        CheckLimit -->|Yes| Wait[명령 배정 대기 및 보존]
        CheckLimit -->|No| PopQueue[우선순위 ZSET 큐 최상단 태스크 POP]
        PopQueue --> CheckLock{작업대 또는 목표 스팟이<br>현재 다른 작업에 점유/락 상태?}
        CheckLock -->|Yes| SkipTask[태스크를 건너뛰고 큐에 유지]
        CheckLock -->|No| FindAMR[IDLE 상태 중 최단거리 AMR 배정]
        FindAMR --> Dispatch[AMR Action Goal 발송 및 active_amr_tasks 1 증가]
    end
```

#### ② JIT 180도 회전 및 일시정지 인터로킹 흐름도
```mermaid
sequenceDiagram
    participant Robot as Sorter/Packaging Robot
    participant CT as Control Tower (ROS 2)
    participant Redis as Redis Priority Queue (ZSET)
    participant AMR as AMR (Mobile Robot)

    Note over Robot, CT: 1. Inbound 4번째 상자 적재 완료 또는 Outbound 4번째 상자 포장 완료
    Robot->>CT: Progress Event (ReportInboundProgress / Packaging Feedback)
    CT->>Robot: Pause Status = True (로봇 일시 정지 발행)
    Note over Robot: 매니퓰레이터 팔 동작 정지 및 안전 대기
    CT->>Redis: Push ROTATE_WORKSTATION (Priority = 100)
    Note over Redis: Task Scheduler가 팝하여 가용한 최단거리 AMR 선정
    CT->>AMR: Send MoveWorkstation Action (ROTATE)
    AMR->>AMR: 물리적 180도 제자리 회전 수행
    AMR->>CT: Action Completed (Succeeded)
    CT->>Robot: Pause Status = False (로봇 일시 정지 해제 발행)
    CT->>CT: Clear rotation_triggered flag
    Robot->>Robot: 5~8번째 슬롯 적재/포장 작업 재개
```

In [ ]:
# 최단거리 AMR 매핑 알고리즘을 모방하는 검증 코드를 수행합니다.
import math

# 출발 타겟 위치 (1번 입고 라인 Active)
start_x, start_y = 7.5, 1.5

# Redis에 올라와 있는 AMR들의 가상 상태
amr_states = {
    "AMR_01": {"x": -6.0, "y": -9.0, "state": "IDLE"},
    "AMR_02": {"x": 6.0, "y": 1.5, "state": "IDLE"},
    "AMR_03": {"x": -3.0, "y": 0.0, "state": "BUSY"}, # 바쁨
    "AMR_04": {"x": 7.5, "y": -3.0, "state": "IDLE"},
    "AMR_05": {"x": 0.0, "y": 9.0, "state": "IDLE"}
}

best_amr = None
min_dist = float("inf")

print(f"🎯 작업 발생지 좌표: ({start_x}, {start_y})")
for amr_id, info in amr_states.items():
    if info["state"] == "IDLE":
        dist = math.sqrt((start_x - info["x"])**2 + (start_y - info["y"])**2)
        print(f"   * {amr_id}: 위치 ({info['x']}, {info['y']}) | 거리: {dist:.2f}m")
        if dist < min_dist:
            min_dist = dist
            best_amr = amr_id

print(f"\n🏆 최단 거리 배정 대상 로봇: {best_amr} (거리: {min_dist:.2f}m)")

---
### 📌 4.3 Dashboard Server 내 SQL 및 캐싱 관리
* **정적 격자 데이터 캐싱 (`grid_cells`)**: 2D 맵을 구성하는 143개의 물리 바닥 격자 및 스팟 정보는 변경되지 않는 정적 데이터이므로 기동 시 최초 1회만 DB에서 조회해 메모리에 캐싱하여, 1.5초마다 발생하는 웹소켓 브로드캐스팅 시 DB 조회 오버헤드를 0%로 단축.
* **트랜잭션 래핑 API**: `/api/upload_packages` (CSV 업로드 시 단일 트랜잭션 내 대량 데이터 INSERT), `/api/start_next_day` (날짜 전환 및 이월 작업대 배치 변경) 등의 API에서 SQL Transaction Commit/Rollback을 안전하게 처리.


## 💻 4. FastAPI 웹 대시보드 서버 (`dashboard_server.py`)

대시보드는 웹소켓을 통한 실시간 정보 분배 및 DOM 성능 렉 개선을 이룬 중앙 모니터링 허브입니다.

### 📌 4.1 실시간 웹소켓 양방향 통신 시퀀스 흐름

```mermaid
graph TD
    Client[웹 대시보드 브라우저] -->|1. WebSocket 연결 요청 /ws| Dash[FastAPI 백엔드]
    Dash -->|2. Connection 수락 및 리스트 저장| ConnMgr[Connection Manager]
    Timer[1.5초 주기 브로드캐스트 루프] -->|3. PostgreSQL/Redis에서 최신 데이터 조회| Dash
    Dash -->|4. 데이터 JSON 직렬화| ConnMgr
    ConnMgr -->|5. 모든 연결된 클라이언트에 send_text| Client
    Client -->|6. CSS absolute 포지셔닝으로 즉시 렌더링| Client
```

### 📌 4.2 absolute 포지셔닝 기반 경량 렌더링
* **기존 문제**: 2D 캔버스에 수많은 객체를 매 프레임 그리면 브라우저 싱글 스레드 렉이 심해져 화면 제어가 불가능했습니다.
* **해결책**: HTML/CSS 바둑판 배경판 하나만 띄워둔 후, **상대 좌표 변환 공식**을 이용해 31개의 고정 설비와 이동하는 로봇들의 top/left % 좌표만 스타일시트로 동적 조절합니다.
  * **변환 수식**:
    * $Grid_X = 50 + (x \times 5.7)\%$
    * $Grid_Y = 50 - (y \times 5.0)\%$
  * DOM 객체 개수가 크게 줄어 렉 현상이 완전히 사라졌습니다.

In [ ]:
# Redis에 저장된 시스템 플래그 값과 대시보드 락 상태를 검증합니다.
import redis
import os

redis_host = os.environ.get("REDIS_HOST", "localhost")
try:
    r = redis.Redis(host=redis_host, port=6379, decode_responses=True)
    
    # 1. 영업 시작 버튼 해제 조건 점검
    csv_status = r.get("system:csv_loaded") == "true"
    
    # 연결된 AMR 기기 스캔
    amr_keys = r.keys("amr:AMR_*")
    online_amrs = []
    for key in amr_keys:
        state = r.hget(key, "state")
        if state:
            online_amrs.append(key.split(":")[1])
            
    print("🔌 대시보드 영업 기동 조건 자가진단:")
    print(f"   * CSV 파일 업로드 상태: {'초록색 (정상)' if csv_status else '빨간색 (미업로드)'}")
    print(f"   * 실시간 연동 감지된 AMR 대수: {len(online_amrs)}대 ({', '.join(online_amrs) if online_amrs else '없음'})")
    
    is_start_button_unlocked = csv_status and (len(online_amrs) > 0)
    print(f"\n📢 결과: [영업 시작] 버튼 비활성화 해제 가능 여부 -> {'🔓 활성화 (START 가능)' if is_start_button_unlocked else '🔒 비활성화 (락 잠금)'}")
    
except Exception as e:
    print(f"❌ Redis 연결 불가: {e}")